In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import StackingClassifier, RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
import pickle

In [2]:
df = pd.read_csv('preprocessed_diabetes_dataset.csv')

X = df.drop(columns=['diabetic'])
y = df['diabetic']

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

In [4]:
# Base models
rf = RandomForestClassifier(
    n_estimators=300, max_depth=20, class_weight='balanced',
    random_state=42, n_jobs=-1
)
gbm = GradientBoostingClassifier(
    n_estimators=300, learning_rate=0.05, random_state=42
)

xgb = XGBClassifier(
    n_estimators=400, learning_rate=0.05, max_depth=6,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric='logloss', random_state=42
)

In [5]:
# Stacking ensemble
stack_model = StackingClassifier(
    estimators=[('rf', rf), ('gbm', gbm), ('xgb', xgb)],
    final_estimator=LogisticRegression(),
    stack_method='predict_proba',
    n_jobs=-1
)


In [6]:
stack_model.fit(X_train, y_train)
y_pred = stack_model.predict(X_test)

with open("diabetes_model.pkl", "wb") as file:
    pickle.dump(stack_model, file)